In [4]:
import pandas as pd
from scipy import stats

# Load baseline cleaned dataset
df = pd.read_csv('HealthConnect_Appointment_Data_Cleaned.csv')

# Standardize column names to match analysis logic
df = df.rename(columns={
    'is_noshow': 'no_show',
    'booking_lead_days': 'lead_time_days'
})

# Verify baseline metrics
print(f"Rows: {len(df)}")
print(f"No-show rate: {df['no_show'].mean() * 100:.2f}%")

Rows: 5000
No-show rate: 48.46%


In [5]:
# Chi-square test for reminders vs no-show
table = pd.crosstab(df['reminder_sent'], df['no_show'])
chi2, p_chi2, _, _ = stats.chi2_contingency(table)

# T-tests for lead time and wait time
t_lead, p_lead = stats.ttest_ind(
    df[df['no_show'] == 0]['lead_time_days'], 
    df[df['no_show'] == 1]['lead_time_days']
)

t_wait, p_wait = stats.ttest_ind(
    df[df['no_show'] == 0]['waiting_time_minutes'], 
    df[df['no_show'] == 1]['waiting_time_minutes']
)

print(f"Reminder Chi2: {chi2:.2f} (p={p_chi2:.4e})")
print(f"Lead Time T-stat: {t_lead:.2f} (p={p_lead:.4e})")
print(f"Wait Time T-stat: {t_wait:.2f} (p={p_wait:.4e})")

Reminder Chi2: 6.30 (p=1.2048e-02)
Lead Time T-stat: -20.02 (p=8.2852e-86)
Wait Time T-stat: -0.07 (p=9.4792e-01)


In [6]:
# Bin lead time and age into categories
df['lead_time_bracket'] = pd.cut(
    df['lead_time_days'], 
    bins=[-1, 7, 21, 45, 100], 
    labels=['0-7 Days', '8-21 Days', '22-45 Days', '45+ Days']
)

df['age_group'] = pd.cut(
    df['age'], 
    bins=[17, 35, 55, 100], 
    labels=['18-35', '36-55', '56+']
)

# Flag wait times over 30 mins
df['high_wait_flag'] = (df['waiting_time_minutes'] > 30).astype(int)

# Export feature matrix for DS model handoff
df.to_csv('HealthConnect_Feature_Engineered_Segments.csv', index=False)